# VN Live-Commerce Host — Colab Bootstrap

Clone the repo → install deps → download model weights → move into the repo layout → run the production `core` `/api/v1` backend → expose via ngrok.

**Architecture:** this backend is the CONTROL plane (JSON + WebSocket). The avatar VIDEO flows LiveAvatar-cloud → LiveKit → browser directly (media plane) — frames never transit this server. Give the printed ngrok URL to `liveavatar_api/frontend/lite.html`.

**Renderer:** `RENDER_BACKEND=cloud` (LiveAvatar) for now; `self_host` is a future diffusion renderer (same API).

Runtime: pick a **GPU** runtime (T4 is fine for gemma-3-4b / Qwen3-4B Q4 + VieNeu-TTS).

## 1. Clone the repo

In [ ]:
# Set your repo URL (HTTPS or with a token for private repos).
REPO_URL = "https://github.com/<you>/<repo>.git"  # <-- EDIT
REPO_DIR = "/content/repo"
IMPL_DIR = REPO_DIR + "/projects/ai-livestream-commerce-vn/implementations"

import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    print("repo already cloned")
%cd $IMPL_DIR
!ls

## 2. Install dependencies

In [ ]:
# Core backend deps + model stacks. llama.cpp (GGUF) for the LLM keeps VRAM low on T4.
!pip install -q -r liveavatar_api/requirements.txt
!pip install -q huggingface_hub llama-cpp-python pyngrok
# TTS: VieNeu-TTS (Apache-2.0, VN-native). Adjust per the model card.
!pip install -q vieneu-tts || echo 'install VieNeu per its model card if the pip name differs'

## 3. Download model weights (LLM + TTS)

LLM is selectable: `gemma-3-4b-it` (Gemma terms) or `Qwen3-4B` (Apache-2.0). We pull a **Q4_K_M GGUF** for llama.cpp. TTS = VieNeu-TTS-v2 (Apache-2.0).

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import os

# --- LLM (GGUF Q4_K_M) ---
# Option A (Apache-2.0): Qwen3-4B GGUF.   Option B: gemma-3-4b-it GGUF (Gemma terms, gated).
LLM_REPO = os.environ.get("LLM_GGUF_REPO", "Qwen/Qwen3-4B-GGUF")
LLM_FILE = os.environ.get("LLM_GGUF_FILE", "")  # e.g. 'Qwen3-4B-Q4_K_M.gguf' (check repo file list)

llm_path = None
if LLM_FILE:
    llm_path = hf_hub_download(repo_id=LLM_REPO, filename=LLM_FILE, local_dir="/content/weights/llm")
    print("LLM GGUF:", llm_path)
else:
    print("Set LLM_GGUF_FILE to the exact Q4_K_M filename from", LLM_REPO)

# --- TTS (VieNeu-TTS-v2, Apache-2.0) ---
TTS_REPO = os.environ.get("TTS_REPO", "pnnbao-ump/VieNeu-TTS-v2")
tts_dir = snapshot_download(repo_id=TTS_REPO, local_dir="/content/weights/tts")
print("TTS dir:", tts_dir)

## 4. Move weights into the repo layout

Place weights where the backend expects them (`implementations/weights/`), so the model loaders resolve a stable path regardless of HF cache location.

In [ ]:
import shutil, os
os.makedirs("weights/llm", exist_ok=True)
os.makedirs("weights/tts", exist_ok=True)

if llm_path:
    dst = os.path.join("weights/llm", os.path.basename(llm_path))
    if not os.path.exists(dst):
        shutil.copy(llm_path, dst)
    print("LLM ->", dst)

# TTS: copy the snapshot tree once
if os.path.isdir(tts_dir) and not os.listdir("weights/tts"):
    shutil.copytree(tts_dir, "weights/tts", dirs_exist_ok=True)
print("weights/:", os.listdir("weights"))

## 5. Set environment (secrets + backend selection)

`LIVEAVATAR_API_KEY` is a **backend-only secret** — never sent to the browser. Use Colab Secrets (🔑 left sidebar) and `userdata.get` to avoid hardcoding.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["LIVEAVATAR_API_KEY"] = userdata.get("LIVEAVATAR_API_KEY")
    os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")
except Exception as e:
    print("Set secrets manually if not on Colab:", e)
    # os.environ["LIVEAVATAR_API_KEY"] = "..."
    # os.environ["NGROK_AUTHTOKEN"] = "..."

os.environ["RENDER_BACKEND"] = "cloud"   # cloud | self_host(future)
os.environ["SESSION_STORE"] = "memory"   # memory(Colab) | redis(AWS)
os.environ["PORT"] = "8800"
print("env set. key loaded:", bool(os.environ.get("LIVEAVATAR_API_KEY")))

## 6. Run the backend + ngrok tunnel

Edit `colab_deploy.build_llm` / `build_tts` to point at the downloaded weights (llama.cpp for the GGUF, VieNeu for TTS). Then this serves `core.server:app` and prints the public URL.

In [ ]:
# Quick sandbox sanity check FIRST (free, no credits): proves the API + key work
!python -m core.tests.v1_smoke_test

In [ ]:
# Launch: loads models, injects into the cloud RenderBackend, serves /api/v1, opens ngrok.
# (This cell blocks and keeps the tunnel alive.)
!python -m liveavatar_api.examples.colab_deploy

## 7. Connect the frontend

Open `liveavatar_api/frontend/lite.html` (locally or any static host), paste the ngrok URL as **Backend URL**, click **Start session**. The avatar video renders via LiveKit; type a viewer message → it routes through `/api/v1/lite/say` → your LLM+TTS → the avatar speaks.